# 7장. RAG 시스템 평가 — 01. RAGAS 합성 테스트 데이터셋 생성

## RAG 시스템의 품질을 어떻게 측정하고 개선할까요?

RAG(Retrieval-Augmented Generation) 시스템을 구축했다면 이제 "잘 동작하는지" 확인해야 해요. 그런데 평가하려면 **테스트 데이터셋**이 먼저 필요합니다. 수백 개의 질문-답변 쌍을 손으로 작성하는 건 현실적이지 않죠.

이 노트북에서는 **RAGAS(Retrieval Augmented Generation Assessment)**를 활용해 문서에서 자동으로 테스트 데이터셋을 생성하는 방법을 다뤄요.

### 학습 목표

- RAGAS의 합성 데이터 생성 파이프라인 이해
- `TestsetGenerator`로 다양한 유형의 질문 자동 생성
- 생성된 데이터셋을 저장하고 다음 평가 단계에서 활용

### 사전 지식

- 6장 RAG 시스템 구축 (문서 로딩, 청킹, 벡터 검색)
- `langchain_community`의 문서 로더 사용법

### 7장 전체 구성

```
01. 테스트 데이터셋 생성 (RAGAS)     ← 지금 여기
02. RAGAS 4가지 지표로 RAG 평가
03. LangSmith 데이터셋과 LLM-as-Judge
04. 커스텀 평가자 구현
05. Heuristic 평가 (ROUGE, BLEU, METEOR)
06. Groundedness(근거성) 평가
07. 모델 비교 평가
08. 온라인 평가 (Online Evaluation)
```

## 합성 데이터 생성의 장점

수동 작성 대신 LLM으로 테스트 데이터를 만들면 어떤 이점이 있을까요?

- **시간 절약**: 수동 작성 대비 대부분의 시간을 절감
- **다양성**: 단순/추론/다중 컨텍스트 등 다양한 난이도의 질문 자동 생성
- **일관성**: 동일한 기준으로 균일한 품질의 데이터셋 구축
- **확장성**: 문서가 바뀌어도 즉시 새 데이터셋 생성 가능

### RAGAS 데이터 생성 흐름

```mermaid
flowchart TD
    A[📄 원본 문서<br/>attention_paper.pdf] --> B[청킹<br/>RecursiveCharacterTextSplitter]
    B --> C[핵심 구문 추출<br/>KeyphraseExtractor]
    C --> D{질문 유형<br/>분배 결정}
    D -->|30%| E[Simple<br/>단순 사실 확인]
    D -->|30%| F[Reasoning<br/>추론 필요]
    D -->|20%| G[Multi-context<br/>여러 청크 통합]
    D -->|20%| H[Conditional<br/>조건부 질문]
    E --> I[Critic LLM<br/>품질 검증]
    F --> I
    G --> I
    H --> I
    I --> J[✅ 테스트셋<br/>question / ground_truth / contexts]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A input
    class B,C,I process
    class D,E,F,G,H process
    class J output
```

---

## 환경 설정

먼저 필요한 패키지를 설치하고 API 키를 설정해요.

In [ ]:
# 필요한 패키지 설치
# RAGAS는 특정 LangChain 버전과 호환됩니다
# !pip install -qU langchain==0.2.16 ragas==0.1.19


In [ ]:
import langchain
import ragas

print(f"LangChain Version: {langchain.__version__}")
print(f"RAGAS Version: {ragas.__version__}")


In [ ]:
from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()


---

## 문서 준비

이 실습에서는 "Attention is All You Need" 논문을 사용해 합성 데이터셋을 생성합니다. Transformer 아키텍처를 소개한 논문으로, 기술적 질문을 생성하기에 내용이 풍부해요.

> **실무 팁**: RAGAS는 `metadata`의 `filename` 키로 같은 문서에 속한 청크를 묶어요. 이 키가 없으면 multi-context 질문 생성에 문제가 생깁니다.

In [ ]:
import os

# ---------------------------------------------------
# 데이터 파일 경로 확인
# ---------------------------------------------------
file_path = "data/attention_paper.pdf"
if not os.path.exists(file_path):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}\n현재 디렉토리: {os.getcwd()}")

In [ ]:
# ---------------------------------------------------
# PDF 문서 로드 및 페이지 수 확인
# ---------------------------------------------------
from langchain_community.document_loaders import PDFPlumberLoader

# PDFPlumberLoader: 표, 레이아웃 정보까지 추출하는 PDF 로더
loader = PDFPlumberLoader("data/attention_paper.pdf")

# 문서 로딩 — 각 페이지가 Document 객체 1개로 변환됨
docs = loader.load()

print(f"총 페이지 수: {len(docs)}")

In [ ]:
# 첫 번째 페이지 내용 확인
print(f"첫 페이지 길이: {len(docs[0].page_content)} 문자")
print(f"\n내용 미리보기:")
print(docs[0].page_content[:300])


### 메타데이터 설정

RAGAS는 문서의 `metadata`에서 `filename` 키를 사용하여 동일한 문서에 속한 청크를 식별합니다. 따라서 각 문서에 `filename` 메타데이터를 추가해야 합니다.


In [ ]:
# 메타데이터 확인
print("현재 메타데이터:")
print(docs[0].metadata)


In [ ]:
# ---------------------------------------------------
# RAGAS 필수 메타데이터 추가: filename 키 설정
# ---------------------------------------------------
# ============================================================
# TODO: 모든 docs에 filename 메타데이터를 추가하세요
# 힌트: for 루프로 doc.metadata["filename"] = "attention_paper.pdf" 설정
# 예상 결과: docs[0].metadata에 'filename' 키가 추가되어 출력
# ============================================================



---

## RAGAS 테스트셋 생성기 설정

테스트셋을 만들려면 세 가지 컴포넌트(구성 요소)가 필요해요.

### 1단계: 필요한 컴포넌트 초기화

RAGAS 테스트셋 생성을 위해 다음 세 컴포넌트가 필요해요.

| 컴포넌트 | 역할 | 권장 설정 |
|---------|------|---------|
| **Generator LLM** | 질문과 답변 초안 생성 | temperature 높게 (다양성) |
| **Critic LLM** | 생성된 질문의 품질 검증 | temperature 0 (일관성) |
| **Embeddings** | 문서 벡터화 및 유사도 계산 | text-embedding-3-small |

> **주의**: Generator와 Critic에 같은 모델을 써도 되지만, 역할이 다르므로 temperature 설정을 다르게 하는 게 좋아요.

In [ ]:
# ---------------------------------------------------
# RAGAS 핵심 컴포넌트 초기화 (Generator / Critic / Embeddings)
# ---------------------------------------------------
# ============================================================
# TODO: Generator LLM, Critic LLM, Embeddings를 초기화하세요
# 힌트: ChatOpenAI(model="gpt-4o-mini", temperature=...)
#       Generator는 temperature=0.7, Critic은 temperature=0
# 예상 결과: 세 객체가 초기화되어 출력
# ============================================================

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# ⚠️ LangChain 1.0에서 langchain.text_splitter는 langchain_text_splitters로 변경되었습니다
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter



### 2단계: DocumentStore(문서 저장소) 설정

`DocumentStore`는 문서를 청크로 분할하고 핵심 구문(keyphrase)을 추출하는 역할을 해요. RAGAS가 다양한 질문을 만들 때 이 구문들을 출발점으로 사용합니다.

In [ ]:
# ---------------------------------------------------
# RAGAS DocumentStore 초기화
# ---------------------------------------------------
# ============================================================
# TODO: DocumentStore를 초기화하세요
# 힌트: InMemoryDocumentStore(splitter=..., embeddings=..., extractor=...)
# 예상 결과: "DocumentStore 초기화 완료" 출력
# ============================================================

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset.extractor import KeyphraseExtractor
from ragas.testset.docstore import InMemoryDocumentStore



### 3단계: TestsetGenerator 생성


In [ ]:
# ---------------------------------------------------
# TestsetGenerator 초기화
# ---------------------------------------------------
from ragas.testset.generator import TestsetGenerator

# TestsetGenerator: Generator LLM + Critic LLM + Embeddings + DocumentStore를 결합
generator = TestsetGenerator.from_langchain(
    generator_llm=generator_llm,
    critic_llm=critic_llm,
    embeddings=ragas_embeddings,
    docstore=docstore
)

print("TestsetGenerator 초기화 완료")

---

## 질문 유형별 분포 설정

RAGAS는 네 가지 유형의 질문을 생성할 수 있어요. 각 유형은 RAG 시스템의 다른 능력을 검사합니다.

| 유형 | 설명 | 검사하는 능력 |
|------|------|-------------|
| **simple** | 단일 청크에서 직접 답변 | 기본 검색 + 생성 |
| **reasoning** | 추론이나 계산 필요 | 복잡한 이해력 |
| **multi_context** | 여러 청크 정보 통합 | 다중 검색 + 통합 |
| **conditional** | 조건부 논리 필요 | 조건 이해력 |

> **실무 팁**: 실제 사용자 질문 분포에 맞게 비율을 조정하세요. 단순 FAQ 봇이라면 simple 비중을 높이고, 리서치 도구라면 reasoning과 multi_context를 높이는 게 좋아요.

In [ ]:
# ---------------------------------------------------
# 질문 유형별 분포 설정 (합계 반드시 1.0)
# ---------------------------------------------------
# ============================================================
# TODO: 4가지 질문 유형(simple, reasoning, multi_context, conditional)의 분포를 설정하세요
# 힌트: distributions = {simple: 0.3, reasoning: 0.3, multi_context: 0.2, conditional: 0.2}
# 예상 결과: 각 유형별 비율이 출력
# ============================================================

from ragas.testset.evolutions import simple, reasoning, multi_context, conditional



---

## 테스트셋 생성

모든 설정이 완료되었어요. 이제 실제로 테스트셋을 생성합니다. 내부적으로는 다음 순서로 처리돼요.

1. 문서를 청크로 분할
2. 각 청크에서 핵심 구문 추출
3. 지정된 분포에 따라 질문 유형 선택
4. Generator LLM이 질문과 정답 생성
5. Critic LLM이 품질 검증 후 필터링

> **주의**: 전체 문서(15페이지)를 처리하면 상당한 시간이 걸려요. 실습에서는 처음 5페이지만 사용합니다.

In [ ]:
# ---------------------------------------------------
# 합성 테스트셋 생성 (내부 처리 순서: 청킹 → 키워드 추출 → 질문 생성 → 품질 검증)
# ---------------------------------------------------
# ============================================================
# TODO: generate_with_langchain_docs()를 호출해 테스트셋을 생성하세요
# 힌트: generator.generate_with_langchain_docs(documents=docs[:5], test_size=15, distributions=distributions, ...)
# 예상 결과: "테스트셋 생성 완료!" 출력
# ============================================================



---

## 생성된 테스트셋 확인

In [ ]:
# ---------------------------------------------------
# 생성된 테스트셋을 DataFrame으로 변환하여 구조 확인
# ---------------------------------------------------
test_df = testset.to_pandas()

print(f"생성된 테스트 케이스 수: {len(test_df)}")
print(f"\n컬럼: {list(test_df.columns)}")

test_df.head()

### 데이터셋 구조

생성된 데이터셋은 다음 컬럼을 포함해요.

| 컬럼 | 설명 | 다음 노트북에서 역할 |
|------|------|-------------------|
| `question` | 생성된 질문 | RAG 시스템에 입력 |
| `ground_truth` | 참조 정답 | Context Recall 계산 기준 |
| `contexts` | 관련 문서 청크 리스트 | Context Precision 계산 기준 |
| `evolution_type` | 질문 유형 | 유형별 성능 분석 |

In [ ]:
# 첫 번째 테스트 케이스 상세 확인
print("=" * 80)
print("📋 테스트 케이스 예시")
print("=" * 80)

sample = test_df.iloc[0]

print(f"\n❓ 질문:")
print(sample['question'])

print(f"\n✅ 정답:")
print(sample['ground_truth'])

print(f"\n📄 관련 컨텍스트 수: {len(sample['contexts'])}")
print(f"\n컨텍스트 1:")
print(sample['contexts'][0][:300] + "...")

print(f"\n🏷️ 질문 유형: {sample.get('evolution_type', 'N/A')}")


### 질문 유형별 분포 확인


In [ ]:
# 질문 유형별 개수 확인
if 'evolution_type' in test_df.columns:
    print("질문 유형별 분포:")
    print(test_df['evolution_type'].value_counts())
    print(f"\n비율:")
    print(test_df['evolution_type'].value_counts(normalize=True).round(2))


---

## 데이터셋 저장

In [ ]:
# ---------------------------------------------------
# 생성된 테스트셋을 CSV 파일로 저장
# ---------------------------------------------------
output_path = "data/attention_synthetic_testset.csv"
test_df.to_csv(output_path, index=False)

print(f"✅ 테스트셋이 저장되었습니다: {output_path}")
print(f"총 {len(test_df)}개의 테스트 케이스")

### 저장 데이터 검증

> **주의**: CSV로 저장하면 `contexts` 컬럼의 리스트가 문자열로 변환돼요. 다음 노트북에서 `ast.literal_eval()`로 다시 리스트로 변환해야 합니다.

In [ ]:
import pandas as pd

# 저장된 파일 다시 읽기
loaded_df = pd.read_csv(output_path)

print(f"로드된 데이터: {len(loaded_df)} rows × {len(loaded_df.columns)} columns")
print(f"\n컬럼: {list(loaded_df.columns)}")

loaded_df.head(3)


---

## 정리

### RAGAS 합성 데이터 생성 6단계 요약

| 단계 | 작업 | 핵심 포인트 |
|------|------|-----------|
| 1. 문서 준비 | PDF 로드 + `filename` 메타데이터 추가 | `filename` 없으면 multi-context 실패 |
| 2. 컴포넌트 설정 | Generator / Critic LLM + Embeddings | Generator temperature 높게 설정 |
| 3. DocumentStore 구성 | 청크 분할 + 키워드 추출 | 청크 크기가 질문 품질에 영향 |
| 4. 질문 분포 정의 | simple/reasoning/multi_context/conditional 비율 | 사용 사례에 맞게 조정 |
| 5. 테스트셋 생성 | `generate_with_langchain_docs()` 호출 | 처음엔 소량으로 테스트 |
| 6. 저장 및 검증 | CSV 저장 + 컬럼 확인 | contexts는 문자열로 저장됨 주의 |

### 다음 노트북 예고

생성한 테스트 데이터셋을 활용해서 RAG 시스템의 성능을 **RAGAS의 4가지 지표**로 평가해볼게요. Faithfulness(충실도), Answer Relevancy(답변 관련성), Context Precision(컨텍스트 정확도), Context Recall(컨텍스트 재현율)이 각각 무엇을 측정하는지 살펴봅니다.